In [1]:
import graphistry
import pandas as pd
%env NX_CURGAPH_AUTOCONFIG=True
import networkx as nx  # <-- Import NetworkX
import matplotlib.pyplot as plt # <-- Import Matplotlib for NetworkX plotting
import numpy as np
graphistry.register(api=3, protocol="https", server="hub.graphistry.com", personal_key_id="XS8DSYS6XO", personal_key_secret="OY126IKROOJDSE8D")

env: NX_CURGAPH_AUTOCONFIG=True


import cudf

import pandas as pd  # Only used for the “zip” part; the rest is on GPU.

# Load the CSV into a cuDF DataFrame:
df_gpu = cudf.read_csv("merged_dedup.csv")

# --- 1. PAPER and SOURCE nodes and published_in edges ---
# Paper nodes: simply use the Title as node_id (assumed unique)
paper_nodes = df_gpu[['Title']].rename(columns={'Title': 'node_id'})
paper_nodes['node_type'] = 'paper'
paper_nodes['node_label'] = paper_nodes['node_id']

# Source nodes:
source_nodes = df_gpu[['Source']].rename(columns={'Source': 'node_id'})
source_nodes['node_type'] = 'source'
source_nodes['node_label'] = source_nodes['node_id']

# published_in edges come directly from the Title and Source columns:
published_edges = df_gpu[['Title', 'Source']].rename(columns={'Title': 'source', 'Source': 'destination'})
published_edges['relationship_type'] = 'published_in'

# --- 2. AUTHORS and authored_by edges ---
# Split the Authors column into a list (cuDF supports string splitting)
df_gpu = df_gpu.assign(authors_list=df_gpu["Authors"].str.split(";"))

# Explode the authors_list so each row becomes one author:
authors_exploded = df_gpu[['Title', 'authors_list']].explode("authors_list")
# Trim whitespace:
authors_exploded['authors_list'] = authors_exploded['authors_list'].str.strip()
# Create the authored_by edges:
authored_by_edges = authors_exploded.rename(columns={'Title': 'source', 'authors_list': 'destination'})
authored_by_edges['relationship_type'] = 'authored_by'

# For each paper, create the position of each author
authors_exploded['author_position'] = authors_exploded.groupby('Title').cumcount()

# Determine the total number of authors for each paper
authors_exploded['total_authors'] = authors_exploded.groupby('Title')['authors_list'].transform('count')

# Mark first and last authors
# First author: position 0
# Last author: position is total_authors - 1
# Others: "coauthor"
authors_exploded['author_type'] = 'coauthor'
authors_exploded['author_type'] = authors_exploded['author_type'].where(authors_exploded['author_position'] != 0, 'fa')
authors_exploded['author_type'] = authors_exploded['author_type'].where(authors_exploded['author_position'] != authors_exploded['total_authors'] - 1, 'mp')

# Create author nodes from these edges:
author_nodes = authors_exploded[['Title', 'authors_list', 'author_type']].drop_duplicates(subset='authors_list')

# Create node_id as the author's name (or another unique identifier if needed)
author_nodes['node_id'] = author_nodes['authors_list']

# Assign the label based on the author type
author_nodes['node_label'] = author_nodes['authors_list'] + " (" + author_nodes['author_type'] + ")"

# Remove duplicate nodes, ensuring each author is unique
author_nodes = author_nodes.drop_duplicates(subset='node_label')

# Display the first few rows for verification:
print(author_nodes.head())

# --- 3. AFFILIATIONS and affiliated_with edges ---
# Process the Affiliations column:
# Replace missing affiliations with an empty string, then split.
df_gpu = df_gpu.assign(affiliations_list = df_gpu["Affiliations"].fillna("").str.split(";"))

# Here comes the trick: we want to pair each author in authors_list with its corresponding affiliation.
# cuDF cannot “zip” two list-columns directly in a vectorized manner.
# One solution is to convert just these two columns to Pandas, apply a row‑wise pairing, and bring the result back to GPU.

# First, select only the columns we need (in CPU memory):
temp_df = df_gpu[["authors_list", "affiliations_list"]].to_pandas()

def pair_authors_affiliations(row):
    """
    Matches authors with affiliations based on available information.
    - If only one affiliation is given, assume all authors share it.
    - If the number of authors and affiliations match, pair them directly.
    - If there are multiple affiliations but fewer than authors, leave assignments blank (since distribution is unknown).
    """
    np.array(row)
    
    authors = [x.strip() for x in row["authors_list"] if isinstance(x, str)]
    affiliations = [x.strip() for x in row["affiliations_list"] if isinstance(x, str) and x.strip() != ""]

    num_authors = len(authors)
    num_affiliations = len(affiliations)

    if num_authors == 0:
        return []  # No authors to pair

    # Case 1: Only one affiliation, assume all authors share it
    if num_affiliations == 1:
        return [(author, affiliations[0]) for author in authors]

    # Case 2: Perfect 1-to-1 match, pair directly
    if num_authors == num_affiliations:
        return list(zip(authors, affiliations))

    # Case 3: More authors than affiliations (but more than one affiliation)
    # → We do NOT know how affiliations are distributed, so we leave assignments blank
    return [(author, "") for author in authors]

# Apply the function to each row:
#pairs_series = temp_df.apply(pair_authors_affiliations, axis=1)
pairs_series_a = df_gpu["authors_list"].apply(pair_authors_affiliations, axis=1)
pairs_series_b = df_gpu["affiliations_list"].apply(pair_authors_affiliations, axis=1)
pairs_series = pairs_series_a.merge(pairs_series_b)

all_pairs = [pair for sublist in pairs_series for pair in sublist]
# Build a Pandas DataFrame with the pairs:
affiliation_edges_pd = pd.DataFrame(all_pairs, columns=['author', 'affiliation'])

# Create affiliated_with edges: here we consider the edge from the author to the affiliation.
affiliation_edges_pd['relationship_type'] = 'affiliated_with'
affiliation_edges_pd['source'] = affiliation_edges_pd['author']
affiliation_edges_pd['destination'] = affiliation_edges_pd['affiliation']

# Create a lookup from affiliation_edges_pd
aff_lookup = affiliation_edges_pd.set_index('author')['affiliation'].to_dict()

# Now, update the exploded authors DataFrame.
# Note: 'authors_exploded' is a cuDF DataFrame, but its string operations are limited.
# For the disambiguation step you can convert to Pandas, process, then re-import to cuDF.
authors_exploded_pd = authors_exploded.to_pandas()

def disambiguate_author(author):
    # Clean the author string.
    key = author.strip()
    # If an affiliation exists and is nonempty, return a composite key.
    if key in aff_lookup and aff_lookup[key].strip() != "":
        return f"{key} ({aff_lookup[key].strip()})"
    return key

# Apply disambiguation to create a new column:
authors_exploded_pd['disambiguated_author'] = authors_exploded_pd['authors_list'].apply(disambiguate_author)

# Now, rebuild the authored_by edges using the disambiguated identifier.
authored_by_edges_pd = authors_exploded_pd.rename(
    columns={'Title': 'source', 'disambiguated_author': 'destination'}
)
authored_by_edges_pd['relationship_type'] = 'authored_by'

# Convert the updated authored_by_edges back to cuDF:
authored_by_edges = cudf.DataFrame(authored_by_edges_pd[['source', 'destination', 'relationship_type']])


# Convert affiliation_edges_pd to a cuDF DataFrame:
affiliation_edges = cudf.DataFrame(affiliation_edges_pd)

# Also, build affiliation nodes from the affiliation column:
affiliation_nodes = affiliation_edges[['destination']].rename(columns={'destination': 'node_id'})
affiliation_nodes['node_type'] = 'affiliation'
affiliation_nodes['node_label'] = affiliation_nodes['node_id']
affiliation_nodes = affiliation_nodes.drop_duplicates(subset='node_id')

# --- 4. Combine All Nodes and Edges ---
nodes_df = cudf.concat([paper_nodes, source_nodes, author_nodes, affiliation_nodes]).drop_duplicates(subset='node_id')
nodes_df=nodes_df[nodes_df["node_label"] != ""]
edges_df = cudf.concat([published_edges, authored_by_edges, affiliation_edges]).drop_duplicates()
ambiguou_edges_df=edges_df.loc[edges_df["destination"]==""]
edges_df =edges_df[edges_df["destination"] != ""]
                     
# Now you have:
#   nodes : a cuDF DataFrame with node_id, node_type, node_label
#   edges : a cuDF DataFrame with source, destination, relationship_type

# Assuming nodes_df and edges_df are already created as cudf.DataFrame

# --- Corrected Plotting the Heterogeneous Graph using bind() ---
g_heterogeneous = graphistry.bind(
    node="node_id",          # Column in nodes_df with node IDs
    source="source",         # Column in edges_df with source node IDs
    destination="destination",    # Column in edges_df with destination node IDs
    edge_label="relationship_type", # Column in edges_df to label edge types
    #node_color="node_type"      # Color nodes by their type (e.g., author, paper)
    # Optional visual customizations can still be added after .plot()
).nodes(nodes_df).edges(edges_df).plot(name="hetero") # Call .nodes() and .edges() with DataFrames and then .plot()

print("\n--- Corrected Graphistry Plot Command for Heterogeneous Graph (g_heterogeneous) created ---")
print("Open g_heterogeneous in a Graphistry notebook or view in browser using g_heterogeneous.url")

import cudf

# Exploding the authors_list so each row becomes one author:
authors_exploded = df_gpu[['Title', 'authors_list']].explode("authors_list")

# Trim whitespace:
authors_exploded['authors_list'] = authors_exploded['authors_list'].str.strip()

# Create the authored_by edges:
authored_by_edges = authors_exploded.rename(columns={'Title': 'source', 'authors_list': 'destination'})
authored_by_edges['relationship_type'] = 'authored_by'

# For each paper, create the position of each author
authors_exploded['author_position'] = authors_exploded.groupby('Title').cumcount()

# Determine the total number of authors for each paper
authors_exploded['total_authors'] = authors_exploded.groupby('Title')['authors_list'].transform('count')

# Mark first and last authors
# First author: position 0
# Last author: position is total_authors - 1
# Others: "coauthor"
authors_exploded['author_type'] = 'coauthor'
authors_exploded['author_type'] = authors_exploded['author_type'].where(authors_exploded['author_position'] != 0, 'fa')
authors_exploded['author_type'] = authors_exploded['author_type'].where(authors_exploded['author_position'] != authors_exploded['total_authors'] - 1, 'mp')

# Create author nodes from these edges:
author_nodes = authors_exploded[['Title', 'authors_list', 'author_type']].drop_duplicates(subset='authors_list')

# Create node_id as the author's name (or another unique identifier if needed)
author_nodes['node_id'] = author_nodes['authors_list']

# Assign the label based on the author type
author_nodes['node_label'] = author_nodes['authors_list'] + " (" + author_nodes['author_type'] + ")"

# Remove duplicate nodes, ensuring each author is unique
author_nodes = author_nodes.drop_duplicates(subset='node_id')

# Display the first few rows for verification:
print(author_nodes.head())


In [ ]:
import cudf
import cugraph
import pandas as pd

print("\n--- Co-authorship Network Analysis (CuGraph) ---")

# --------------------------------------------------------------------------
# A. Generate Co-author Edge List (GPU Version via Self-Join)
# --------------------------------------------------------------------------
# We start with the paper->author edges (i.e. relationship "authored_by").

# Create co-authorship edges from the exploded authors
a = authors_exploded[['Title', 'authors_list']].rename(columns={'authors_list': 'author1'})

# Convert this to cuDF DataFrame
coauthorship_edges = cudf.DataFrame(coauthorship_edges)

# Check the first few rows of coauthorship edges
print(coauthorship_edges.head())

# Filter out self-pairs and duplicate author pairs; enforce an ordering
coauthor_edges_df = coauthorship_edges
# After building coauthor_edges_df
if not isinstance(coauthor_edges_df, cudf.DataFrame):
    coauthor_edges_df = cudf.DataFrame(coauthor_edges_df)

print("\n--- Co-authorship Edges DataFrame (first few rows) ---")
print(coauthor_edges_df.head())

# --------------------------------------------------------------------------
# B. Create the Co-authorship Graph using CuGraph
# --------------------------------------------------------------------------
G_coauthor = cugraph.Graph()
G_coauthor.from_cudf_edgelist(coauthor_edges_df, source='source', destination='destination')

print("\n--- Co-authorship Graph Created (CuGraph) ---")
print(f"Number of nodes in co-authorship graph: {G_coauthor.number_of_vertices()}")
print(f"Number of edges in co-authorship graph: {G_coauthor.number_of_edges()}")

# --------------------------------------------------------------------------
# C. Calculate Degree Centrality in the Co-authorship Graph
# --------------------------------------------------------------------------
# Attempt to use cugraph.degree_centrality. If unavailable, compute degree and normalize.
try:
    coauthor_deg_df = cugraph.degree_centrality(G_coauthor)
except Exception as e:
    print("cugraph.degree_centrality not available; computing degree centrality manually.")
    deg_df = G_coauthor.degree()
    n_vertices = G_coauthor.number_of_vertices()
    # Normalize degree centrality by (n_vertices - 1)
    if n_vertices > 1:
        deg_df['degree_centrality'] = deg_df['degree'] / (n_vertices - 1)
    else:
        deg_df['degree_centrality'] = 0.0
    coauthor_deg_df = deg_df[['vertex', 'degree_centrality']]

print("\n--- Co-author Degree Centrality Calculated (CuGraph) ---")
print(coauthor_deg_df.head())

# --------------------------------------------------------------------------
# D. Merge Co-author Centrality with Author Node DataFrame for Visualization
# --------------------------------------------------------------------------
# Ensure author_nodes_df is on GPU. If it's a Pandas DataFrame, convert it:
author_nodes_cudf = cudf.DataFrame(author_nodes)  # Must contain at least a 'node_id' column
# Merge co-author centrality values from coauthor_deg_df (where 'vertex' = author id)
author_nodes_merged = author_nodes_cudf.merge(
    coauthor_deg_df, left_on='node_id', right_on='vertex', how='left'
).fillna(0)

print("\nTop 10 authors by co-author degree centrality:")

# Rename 'vertex' to 'node_id' in the coauthor_deg_df DataFrame
coauthor_deg_df = coauthor_deg_df.rename(columns={'vertex': 'node_id'})
# Standardize keys if necessary (for instance, trim and lowercase)
author_nodes_cudf["node_id"] = author_nodes_cudf["node_id"].astype("str").str.strip().str.lower()
coauthor_deg_df["node_id"] = coauthor_deg_df["node_id"].astype("str").str.strip().str.lower()

# Perform the merge normally
author_nodes_merged = author_nodes_cudf.merge(coauthor_deg_df, on='node_id', how='left').fillna(0)

# Inspect the merged columns
print("Merged columns:", list(author_nodes_merged.columns))

# Since we see that the merged DataFrame has 'degree_centrality_x' and 'degree_centrality_y',
# decide which one to keep. For example, if you want to keep the values from coauthor_deg_df:
#author_nodes_merged = author_nodes_merged.drop(columns=['degree_centrality_x'])
#author_nodes_merged = author_nodes_merged.rename(columns={'degree_centrality': 'degree_centrality'})

# Now sorting should work:
print(author_nodes_merged.sort_values(by='degree_centrality', ascending=False).head(10))


# --------------------------------------------------------------------------
# E. Basic Network Statistics for the Co-authorship Graph
# --------------------------------------------------------------------------
n_edges = G_coauthor.number_of_edges()
if n_edges > 0:
    n_vertices = G_coauthor.number_of_vertices()
    density_coauthor = (2 * n_edges) / (n_vertices * (n_vertices - 1)) if n_vertices > 1 else 0.0

    # Calculate average degree using degree() output
    deg_df = G_coauthor.degree()
    avg_degree_coauthor = deg_df['degree'].sum() / n_vertices

    print(f"\nCo-authorship Network Density: {density_coauthor:.4f}")
    print(f"Average Degree in Co-authorship Network: {avg_degree_coauthor:.2f}")


# f) Connected Components and Sizes in Co-authorship Network (CuGraph) # More descriptive comment
    components_df_cg = cugraph.weakly_connected_components(G_coauthor) # Use cuGraph function for labels
    print(f"Connected Components Calculated using CuGraph (with labels)")

    comp_sizes_cg = components_df_cg.groupby('labels').agg({'vertex': 'count'}).reset_index() # Group component labels in cuGraph DataFrame
    largest_cc_size = comp_sizes_cg['vertex'].max()
    pct_largest = (largest_cc_size / n_vertices) * 100 if n_vertices > 0 else 0.0
    print(f"Size of Largest Connected Component: {largest_cc_size} authors, representing {pct_largest:.2f}% of authors")
# --------------------------------------------------------------------------
# F. Community Detection on the Co-authorship Graph (Louvain)
# --------------------------------------------------------------------------
if not coauthor_edges_df.empty:
    # Run Louvain community detection and unpack the result if necessary.
    result = cugraph.louvain(G_coauthor)
    if isinstance(result, tuple):
        communities_df, modularity = result
    else:
        communities_df = result
    
    # Now, communities_df should be a DataFrame with at least the columns 'vertex' and 'partition'.
    num_communities = communities_df['partition'].nunique()
    print(f"\n--- Louvain Community Detection (CuGraph) ---")
    print(f"Number of communities detected: {num_communities}")
    
    # Optionally, print the modularity score if available.
    if 'modularity' in locals():
        print(f"Modularity: {modularity}")


    # Summarize community sizes (display top 10 largest)
    community_sizes = communities_df.groupby('partition').agg({'vertex': 'count'}).reset_index()
    community_sizes = community_sizes.sort_values(by='vertex', ascending=False)
    community_sizes_pd = community_sizes.to_pandas()  # For pretty printing on CPU
    print("\n--- Community Size Distribution (Top 10 largest) ---")
    for idx, row in community_sizes_pd.head(10).iterrows():
        print(f"Community {row['partition']}: {row['vertex']} authors")
else:
    print("\nWarning: Co-authorship network has no edges, skipping community detection.")
    communities_df = cudf.DataFrame()  # empty DataFrame

# --------------------------------------------------------------------------
# G. Prepare Community ID DataFrame for Graphistry Visualization
# --------------------------------------------------------------------------
if communities_df.shape[0] != 0:
    # Rename columns: 'vertex' becomes 'node_id' and 'partition' is the community assignment.
    community_id_map = communities_df.rename(columns={'vertex': 'node_id',
                                                       'partition': 'community_id_louvain'})
    author_nodes_with_comm = author_nodes_cudf.merge(community_id_map, on='node_id', how='left')
    # Fill missing community IDs with -1 and convert the type to integer
    author_nodes_with_comm['community_id_louvain'] = author_nodes_with_comm['community_id_louvain'].fillna(-1).astype('int32')
    print("\n--- Author Nodes DataFrame with Louvain Community IDs (for Graphistry) ---")
    print(author_nodes_with_comm.head())
else:
    print("\nWarning: Community detection was skipped or no communities found. Graphistry visualization will not be community-colored.")
    author_nodes_with_comm = author_nodes_cudf.copy()


In [ ]:
import cudf

# Assuming 'author_nodes_with_comm' is a cuDF DataFrame containing authors' node information with 'node_label' (affiliations)
# and 'community_id_louvain' (community labels)

# Extract affiliations (assuming affiliation is in parentheses in the 'node_label')
author_nodes_with_comm['affiliation'] = author_nodes_with_comm['node_label'].str.extract(r'\((.*?)\)', expand=False)

# If no affiliation is found (NaN values), replace them with 'Unknown'
author_nodes_with_comm['affiliation'] = author_nodes_with_comm['affiliation'].fillna('Unknown')

# Group by 'community_id_louvain' and aggregate affiliations by joining them
# Convert each group of affiliations to a list, then join the list into a single string
community_affiliations = author_nodes_with_comm.groupby('community_id_louvain')['affiliation'].apply(lambda x: ', '.join(x.to_arrow().to_pylist())).reset_index()

# Count the size (number of authors) in each community
community_sizes = author_nodes_with_comm.groupby('community_id_louvain').size().reset_index(name='community_size')

# Merge the community affiliations with the sizes
community_affiliations_with_size = community_affiliations.merge(community_sizes, on='community_id_louvain')

# Display the resulting DataFrame
print(community_affiliations_with_size)

import graphistry
import cudf # Ensure cudf is imported

# Assuming coauthor_edges_df, author_nodes_merged are already created as cudf.DataFrame
# and G_coauthor graph is also available

# --- Corrected Plotting the Co-authorship Graph using bind() ---
g_coauthorship = graphistry.bind(
    source="author1",        # Column in coauthor_edges_df for source author
    destination="author2",   # Column in coauthor_edges_df for destination author
    node="node_id",         # Column in author_nodes_merged (or author_nodes_cudf) with author node IDs
    #node_color="degree_centrality", # Color nodes by degree centrality (from author_nodes_merged)
    #node_size="degree_centrality",  # Size nodes by degree centrality (optional - for visual emphasis)
    #node_label="node_name"       # Label nodes with author names (if 'node_name' exists)
    # Optional visual customizations can still be added after .plot()
).nodes(author_nodes_merged).edges(coauthor_edges_df) # Call .nodes(), .edges() and .plot() in chain

print("\n--- Corrected Graphistry Plot Command for Co-authorship Graph (g_coauthorship) created ---")
print("Open g_coauthorship in a Graphistry notebook or view in browser using g_coauthorship.url")
g_coauthorship.plot()


In [1]:
import spacy
import polars as pl
df = pl.read_csv("merged_dedup.csv")

In [2]:
df_exploded = df.with_columns(
    pl.col("Affiliations").str.split(";").alias("Affiliation_list")
).explode("Affiliation_list")


In [3]:
df_exploded

Title,Abstract,Authors,Affiliations,Year,Source,Author Keywords,Cite Count,DOI,Affiliation_list
str,str,str,str,i64,str,str,f64,str,str
"""Deep manifold learning for the…","""Intracranial neural activity r…","""Wu, Lingyun (57226766385); Hu,…","""Department of Neurology, Tangs…",2025,"""Biomed. Signal Process. Contro…","""Deep manifold learning; EEG; I…",0.0,"""10.1016/j.bspc.2024.107335""","""Department of Neurology, Tangs…"
"""Deep manifold learning for the…","""Intracranial neural activity r…","""Wu, Lingyun (57226766385); Hu,…","""Department of Neurology, Tangs…",2025,"""Biomed. Signal Process. Contro…","""Deep manifold learning; EEG; I…",0.0,"""10.1016/j.bspc.2024.107335""",""" School of Electrical and Info…"
"""Enhancing motor imagery task r…","""Motor imagery electroencephalo…","""Dovedi, Tanvi (57215744556); U…","""Department of Electronics and …",2025,"""Biomed. Signal Process. Contro…","""Brain-Computer Interface; Clas…",0.0,"""10.1016/j.bspc.2024.107149""","""Department of Electronics and …"
"""MCMTNet: Advanced network arch…","""Brain–computer interface (BCI)…","""Yang, Yingjie (59493581600); Z…","""Tianjin Key Laboratory of Wire…",2025,"""Neurocomputing""","""Brain–machine interface; Convo…",0.0,"""10.1016/j.neucom.2024.129255""","""Tianjin Key Laboratory of Wire…"
"""MCMTNet: Advanced network arch…","""Brain–computer interface (BCI)…","""Yang, Yingjie (59493581600); Z…","""Tianjin Key Laboratory of Wire…",2025,"""Neurocomputing""","""Brain–machine interface; Convo…",0.0,"""10.1016/j.neucom.2024.129255""",""" College of Artificial Intelli…"
…,…,…,…,…,…,…,…,…,…
"""Quantum Neural Network-Based E…","""A novel neural information pro…","""V. Gandhi; G. Prasad; D. Coyle…","""Middlesex University, School o…",2014,"""IEEE Transactions on Neural Ne…","""Brain–computer interface (BCI)…",87.0,null,"""Middlesex University, School o…"
"""Quantum Neural Network-Based E…","""A novel neural information pro…","""V. Gandhi; G. Prasad; D. Coyle…","""Middlesex University, School o…",2014,"""IEEE Transactions on Neural Ne…","""Brain–computer interface (BCI)…",87.0,null,""" University of Ulster, Intelli…"
"""Quantum Neural Network-Based E…","""A novel neural information pro…","""V. Gandhi; G. Prasad; D. Coyle…","""Middlesex University, School o…",2014,"""IEEE Transactions on Neural Ne…","""Brain–computer interface (BCI)…",87.0,null,""" University of Ulster, Intelli…"


In [131]:
refined_categories = {
    "Medical_Sciences": ["neur", "brain", "cognitive", "sleep", "speech", "vision", "alzheimer", "autismo", "CLLE", "MEGIN", "Mental", "CERMEP", "Cerveau", "CRANIA", "H. Lundbeck", "Hangzhou First People", "Pfeiffersche", "Cirugía", "Médica", "Neonatology", "UZ Leuven", "General Practice", "Mmu", "Médecine", "UKM", "UMSC", "Pediatrics", "UMC", "Hubei Shizhen", "UPC", "CHU", "thera", "med", "Inserm", "patho", "pato", "cancer", "oncology", "radiology", "injury", "human", "health", "psiquiatr", "psychiatr", "hospital", "Hôpital", "clinic", "care", "Anästhesiologie", "Anesthesi", "restorative", "regen", "Otorhinolaryngology", "sports", "epilep", "tremo", "surgery", "Cognition", "Cardiovascular", "Dementia", "Geriatrics", "Diabetes", "ExpORL","Philosophy"],
    "Computer_Science": ["inform","CAPSIX","Cibernetica","LAVID","Fuzzy Inst","ATR","FAST NUCES","CNRS","computer","intel","wireless","tele","systems","ai","smart","Supercomputing","CSE","LIPADE","Data Analysis","Simulation","tech","processing","signal","I.T ","IT ","Data","LTSI","Iasi","Sys","internet","online","BCI","CDAC","Robotics","Zhejiang Lab","ECE","ICT","Ji Hua Laboratory"],
    "Engineering": ["AJCE","Russell","RIKEN","Budapesti","CSIRO","TU ","CEA","ISEP","IOSB","Schmalkalden","EIE","I.I.E.S.T","IIEST","INEGI","VTU","SSGMCE","FNSPE","JISCE","PIGCE","UIET","JNTU","Hubei Longzhong","TU Braunschweig","TU-Sofia","JEC","Hahn-Schickard","Ingénieur","Inesc","Changsha","Design","UNSTPB","Tianmushan","Delft","eng", "technol","ronics","imec","ele","Physique","Grenoble","Akademie","Techn","Polit","Automation","Physics","Mechanical","Mat","bioengineeri","cybernetics","Biochemistry","Biosciences","DIBRIS","Teknologi","Machine","circuit","ULFG","GRIET","Inria","Analog","Innovative","Inovasi" ,"HHI","VDF GOI","DIEMS"],
    "Private_Entity":["Sulc","WDH","Gedeon Richter","plc","Fraunhofer","Noordwest","Precisis","Sentia","MindRove","LG","Greentek","CSIRO Manufacturing","Whist Lab","Daewon Precision","KaosKey","Ambu A/S, Baltorpbakken","THALES","A*STAR","Charles River","LLC","Ullo","AMO ","Foxstream","Google","Siemens","Everseen","Tecent","Tencent","MindAffect","Porsche","OptoCeutics","HypnoVR","Toyota","VASCage","Hollysys","Soso","Hearing4all","Fujitsu","Zeto","Amazon","IBM","Zalando","Senzhigaoke","Limited","Co","ltd","clinic","GmbH","CESI LINEACT","Inc","sa","Alibaba","R&D","WS Audiology","Solarwinds","International","WeBank","trianect","MindSpire","Epilog","MINDig","group","Byteflies","Pre Diagnostics","Orange","qEEG-Pro","Csl","NEXARCH","Global","SimiaRoom"],
    "Interdisciplinary": ["Jamia Hamdard","Jamia Millia Islamia","IFHE","CRC","Korea U","HU Berlin","htw","UNSW","RISE","Zhongguancun","Faculdade","seibersdorf","uni","Interdisciplinary","Binghamton","Innovation","research","sciences","college","univer","Science","Independent","LMU","Industries","Cient","Network","Akademi","Academy","CIFAR","CESI","AAU" ],
    "Participants_Associations": ["Fonctionnelles","Centre","CASE Danismanlik","footpower","Kinderziekenhuis","Clinique","Troop","spc","Dyslexia","Association","EuroRehab","Pilipili","ESSIQ","CBR Effata","Gymnasium","Klinik","allmyhomes","Abled","Museum","Expert","Orderin","Tennis","Accenture","Hospices","Tam Finans Ar-Ge Tam Finans","Orthopedic Prosthetics","high","school","Foundation","Administrative","Bureau","FHI 360","Promotion","Transport","ambulan","Education","Denken Solution","Timeflux","Residency","center","Ministry","child","Planning","ROC Kop van Noord-Holland"],
    "Other": ["Alumni","and","Fellow","Serra Hunter","Thiruvananthapuram","CREST","na","CGM", "HEC","social","music","culture","history","art","psycholog","Psicolo","tourism","Liser","Management","IMI","Ahoura Building","Pharma","Microbiology","Physiology","Cytometry","Kinesiology","child development","Bio","Ageing","Genetic","Guangzhou","Eriksholm","Hefei","oil","BGI","3B","Genomic","fMRI","LTCI"],
}

# --- Proposed Improved Version ---
refined_categories_improved = {
    "Medical_Sciences": [
        "neur", "brain", "cognitive", "sleep", "speech", "vision", "alzheimer", "autismo", "CLLE", "MEGIN", "Mental",
        "CERMEP", "Cerveau", "CRANIA", "H. Lundbeck", "Hangzhou First People", "Pfeiffersche", "Cirugía", "Médica",
        "Neonatology", "UZ Leuven", "General Practice", "Mmu", "Médecine", "UKM", "UMSC", "Pediatrics", "UMC",
        "Hubei Shizhen", "UPC", "CHU", "thera", "med", "Inserm", "patho", "pato", "cancer", "oncology", "radiology",
        "injury", "human", "health", "psiquiatr", "psychiatr", "hospital", "Hôpital","Ospedale", "clinic", "care",
        "Anästhesiologie", "Anesthesi", "restorative", "regen", "Otorhinolaryngology", "sports", "epilep", "tremo",
        "surgery", "Cognition", "Cardiovascular", "Dementia", "Geriatrics", "Diabetes", "ExpORL","PériTox",
        # --- NEWLY ADDED / MOVED FROM 'OTHER' ---
        "psycholog", "Psicolo","Hopital","Apnea","Hearing","Anesteziyoloji","Nutrition","ICM","UPMC","ASL"
        "Physiology", "Cytometry", "Kinesiology", "child development","Black Dog Institute","parkinson","IRCCS", "Don Carlo Gnocchi"
    ],
    "Computer_Science": [
        "inform","CAPSIX","Cibernetica","LAVID","Fuzzy Inst","ATR","FAST NUCES","CNRS","computer","intel","wireless",
        "tele","systems","ai","smart","Supercomputing","CSE","LIPADE","Data Analysis","Simulation","tech","processing",
        "signal","I.T ","IT ","Data","LTSI","Iasi","Sys","internet","online","BCI","CDAC","Robotics","Zhejiang Lab","GGITS","SINTEF","Asim",
        "ECE","ICT","Ji Hua Laboratory","digital society","digital identity","cyber security","JIIT","Bruno Kessler","IIT","VIT","Bits","LIP", "Hk3",

        
    ],
    "Engineering": [
        "AJCE","Russell","RIKEN","Budapesti","CSIRO","TU ","CEA","ISEP","IOSB","Schmalkalden","EIE","I.I.E.S.T","IIEST",
        "INEGI","VTU","SSGMCE","FNSPE","JISCE","PIGCE","UIET","JNTU","Hubei Longzhong","TU Braunschweig","TU-Sofia","JEC",
        "Hahn-Schickard","Ingénieur","Inesc","Changsha","Design","UNSTPB","Tianmushan","Delft","eng", "technol","ronics",
        "imec","ele","Physique","Grenoble","Akademie","Techn","Polit","Automation","Physics","Mechanical","Mat",
        "bioengineeri","cybernetics","Biochemistry","Biosciences","DIBRIS","Teknologi","Machine","circuit","ULFG","CIRMANMEC"
        "GRIET","Inria","Analog","Innovative","Inovasi" ,"HHI","VDF GOI","DIEMS","Helmholtz Institute Ulm","DoEC, GGITS","GMIT","MANIT","Nit",
        "battery", "batteries", "energy storage", "electrochemical", "materials science", "akkumulator", "elektrochemie", "energie","Graphene","PVPIT",
    ],
    "Private_Entity":[
        "Sulc","WDH","Gedeon Richter","plc","Fraunhofer","Noordwest","Precisis","Sentia","MindRove","LG","Greentek",
        "CSIRO Manufacturing","Whist Lab","Daewon Precision","KaosKey","Ambu A/S, Baltorpbakken","THALES","A*STAR",
        "Charles River","LLC","Ullo","AMO ","Foxstream","Google","Siemens","Everseen","Tecent","Tencent","MindAffect",
        "Porsche","OptoCeutics","HypnoVR","Toyota","VASCage","Hollysys","Soso","Hearing4all","Fujitsu","Zeto","Amazon",
        "IBM","Zalando","Senzhigaoke","Limited","Co","ltd","clinic","GmbH","CESI LINEACT","Inc","sa","Alibaba","R&D",
        "WS Audiology","Solarwinds","International","WeBank","trianect","MindSpire","Epilog","MINDig","group",
        "Byteflies","Pre Diagnostics","Orange","qEEG-Pro","Csl","NEXARCH","Global","SimiaRoom","Orobix Life","Empatica","TDK","S.p.A","Spa","srl","s.r.l",
        "Bayer","GMV",

    
    ],
    "Interdisciplinary": [
        "Jamia Hamdard","Jamia Millia Islamia","IFHE","CRC","Korea U","HU Berlin","htw","UNSW","RISE","Zhongguancun",
        "Faculdade","seibersdorf","uni","Interdisciplinary","Binghamton","Innovation","research","sciences","college",
        "univer","Science","Independent","LMU","Industries","Cient","Network","Akademi","Academy","CIFAR","CESI","AAU","Üniversitesi",
        "cnr","Ciência","Campus","PSL","Instit","Umberto Veronesi","URECA","JST"
    ],
    "Participants_Associations": [
        "Fonctionnelles","Centre","CASE Danismanlik","footpower","Kinderziekenhuis","Clinique","Troop","spc","Dyslexia",
        "Association","EuroRehab","Pilipili","ESSIQ","CBR Effata","Gymnasium","Klinik","allmyhomes","Abled","Museum",
        "Expert","Orderin","Tennis","Accenture","Hospices","Tam Finans Ar-Ge Tam Finans","Orthopedic Prosthetics",
        "high","school","Foundation","Administrative","Bureau","FHI 360","Promotion","Transport","ambulan","Education",
        "Denken Solution","Timeflux","Residency","center","Ministry","child","Planning","ROC Kop van Noord-Holland",
        "Charité","charity","Crossing Dialogues","Alliance","IRHIS",
    ],
    "Other": [
        "Alumni","and","Fellow","Serra Hunter","Thiruvananthapuram","CREST","na","CGM", "HEC","Philosophy","social",
        "music","culture","history", "art", "tourism","Liser","Management","IMI","Ahoura Building",
        "Pharma","Microbiology", # Removed: "Physiology","Cytometry","Kinesiology","child development","psycholog", "Psicolo"
        "Bio","Ageing","Genetic", # "Bio" remains here
        "Guangzhou","Eriksholm","Hefei","oil","BGI","3B","Genomic","fMRI","LTCI"
    ],
}

In [262]:
refined_categories_improved_sorted = {
    "Medical_Sciences": [
        # Medical Conditions / Specializations
        "alzheimer", "Anästhesiologie", "Anesthesi", "Apnea", "autismo", "cancer", "Cardiovascular", "Cerveau",
        "Cognition", "cognitive", "Dementia", "Diabetes", "epilep", "Geriatrics", "Hearing", "health", "human",
        "injury", "Mental", "Neonatology", "Nutrition", "oncology", "Otorhinolaryngology", "parkinson", "patho",
        "Pediatrics", "PériTox", "Physiology", "psiquiatr", "psychiatr", "psycholog", "Psicolo", "radiology",
        "sleep", "speech", "sports", "surgery", "tremo", "vision",

        # Medical Institutions / Labs / Initiatives (Specific)
        "Anesteziyoloji", "ASL", "Black Dog Institute", "CERMEP", "CHU", "clini", "CRANIA", "ExpORL",
        "General Practice", "H. Lundbeck", "Hangzhou First People", "Hopital", "hospital", "Hôpital",
        "Hubei Shizhen", "ICM", "Inserm", "IRCCS", "Mmu", "Médecine", "med", "MEGIN", "Ospedale", "pato",
        "Pfeiffersche", "thera", "UKM", "UMC", "UMSC", "UPC", "UPMC", "UZ Leuven",
        "Don Carlo Gnocchi", # Moved to specific institutions/initiatives

        # Medical Practices / Care
        "care", "Cirugía", "Médica", "regen", "restorative",

        # Broader Medical/Life Science Fields (less specific than conditions/specialties)
        "brain", "Cytometry", "Kinesiology", "neur",
        "child development", # kept for clarity, also fits here
        "CLLE", # Acronym, could be specific inst. or project
    ],
    "Computer_Science": [
        # Core Computer Science / AI / Data Terms
        "ai", "computer", "cyber security", "Data", "Data Analysis", "digital identity", "digital society",
        "inform", "internet", "I.T ", "IT ", "online", "processing", "Robotics", "signal", "Simulation",
        "smart", "Supercomputing", "Sys", "tech", "tele", "wireless", "systems",

        # Computer Science Institutions / Labs / Initiatives (Specific)
        "Asim", "ATR", "BCI", "Bruno Kessler", "CAPSIX", "CDAC", "Cibernetica", "CNRS", "CSE", "ECE",
        "FAST NUCES", "GGITS", "Hk3", "ICT", "IIT", "Iasi", "Ji Hua Laboratory", "JIIT", "LAVID", "LIP", "LIPADE",
        "LTSI", "SINTEF", "VIT", "Zhejiang Lab", "Bits", "Fuzzy Inst", # Fuzzy Inst is an org name
    ],
    "Engineering": [
        # Core Engineering / Physics / Materials Terms
        "Analog", "Automation", "battery", "batteries", "Biochemistry", "bioengineeri", "Biosciences", "circuit",
        "cybernetics", "Design", "ele", "electrochemical", "eng", "energie", "energy storage", "Graphene",
        "Ingénieur", "Machine", "Mat", "materials science", "Mechanical", "Physics", "Physique", "Polit",
        "ronics", "Techn", "technol", "Teknologi", "akkumulator", "elektrochemie",

        # Engineering Institutions / Labs / Initiatives (Specific)
        "AJCE", "Akademie", "Changsha", "CIRMANMEC", "CSIRO", "Delft", "DIBRIS", "DIEMS", "DoEC, GGITS",
        "EIE", "FNSPE", "GRIET", "Grenoble", "Hahn-Schickard", "Helmholtz Institute Ulm", "HHI", "I.I.E.S.T",
        "IIEST", "imec", "INEGI", "Inesc", "Innovative", "Inovasi", "Inria", "ISEP", "IOSB", "JEC", "JISCE",
        "JNTU", "MANIT", "Nit", "PIGCE", "PVPIT", "RIKEN", "Russell", "Schmalkalden", "Tianmushan", "TU ",
        "TU Braunschweig", "TU-Sofia", "UIET", "ULFG", "UNSTPB", "VDF GOI", "VTU", "Budapesti", "GMIT",
        "Hubei Longzhong", "SSGMCE", # These are likely institutions/universities
    ],
    "Private_Entity": [
        # Company Types / Legal Designations
        "Co", "GmbH", "Inc", "International", "Limited", "LLC", "ltd", "plc", "R&D", "sa", "S.p.A", "Spa",
        "srl", "s.r.l", "group",

        # Specific Company Names
        "STAR","Accenture","Akvaplan", "Alibaba", "AMO ", "Amazon", "Ambu A/S, Baltorpbakken", "Bayer", "Byteflies", "CESI LINEACT",
        "Charles River", "Csl", "CSIRO Manufacturing", "Daewon Precision", "Dagger", "Epilog", "Empatica",
        "Everseen", "Foxstream", "Fraunhofer", "Fujitsu", "Gedeon Richter", "Global", "GMV", "Google",
        "Greentek", "Hospices", "Hollysys", "Hearing4all", "HHI", "HypnoVR", "IBM","Intel", "KaosKey", "LG", "MindAffect",
        "MindRove", "MindSpire", "MINDig", "NEXARCH", "Noordwest", "OptoCeutics", "Orange", "Orderin",
        "Orobix Life", "Porsche", "Pre Diagnostics", "Precisis", "qEEG-Pro", "Sentia", "Siemens", "SimiaRoom",
        "Solarwinds", "Soso", "Sulc", "Tam Finans Ar-Ge Tam Finans", "TDK", "Tecent", "Tencent", "THALES",
        "Timeflux", "trianect", "Toyota", "Ullo", "VASCage", "WDH", "WeBank", "Whist Lab", "WS Audiology",
        "Zalando", "Zeto", "Senzhigaoke",

        # General Private/Commercial Terms (often indicate a private company)
        "clinic", # Can be private or public, but often used for private clinics.
    ],
    "Interdisciplinary": [
        # General Academic / Research Terms
        "Akademi", "Academy", "Campus", "Ciência", "Cient", "college", "Independent", "Industries", "Innovation",
        "Instit", "Network", "research", "Science", "sciences", "uni", "univer", "Üniversitesi",

        # Specific Interdisciplinary Institutions / Centers
        "AAU", "Binghamton", "CESI", "CIFAR", "cnr", "CRC", "Faculdade", "HU Berlin", "htw","ifhe", "Jamia Hamdard",
        "Jamia Millia Islamia", "JST", "Korea U", "LMU", "PSL", "RISE", "seibersdorf", "Umberto Veronesi",
        "UNSW", "URECA", "Zhongguancun",
    ],
    "Participants_Associations": [
        # Organizational Types / Missions
        "Administrative", "Alliance", "Association", "Bureau", "Centre", "charity", "child", "Clinic",
        "Crossing Dialogues", "Education", "Expert", "FHI 360", "Fonctionnelles", "Foundation", "Gymnasium",
        "high", "Hospices", "Kinderziekenhuis", "Klinik", "Ministry", "Museum", "Planning", "Promotion",
        "Residency", "school", "spc", "Troop", "center",

        # Specific Organizations / Initiatives (People-focused, social, advocacy)
        "Abled", "allmyhomes", "ambulan", "CASE Danismanlik", "CBR Effata", "Charité", "Denken Solution",
        "Dyslexia", "ESSIQ", "EuroRehab", "footpower", "IRHIS", "Orthopedic Prosthetics", "Pilipili",
        "ROC Kop van Noord-Holland", "Tennis", "Transport",
    ],
    "Other": [
        # Academic Status / Affiliations (often generic or partial)
        "Alumni", "and", "Fellow", "na",

        # Geographic / Location / Generic Identifiers (that aren't institutions themselves)
        "Ahoura Building", "3B", "CGM", "CREST", "Eriksholm", "fMRI", "Guangzhou", "Hefei", "HEC", "IMI",
        "Liser", "LTCI", "oil", "Thiruvananthapuram",

        # Broad Scientific / Research Terms (that are often too generic or not primary categories)
        "Ageing", "Bio", "BGI", "Genetic", "Genomic", "Pharma", "Microbiology",

        # Humanities / Arts / Social Sciences (Non-Medical/Tech/Eng)
        "art", "culture", "history", "Management", "music", "Philosophy", "social", "tourism",
        "Serra Hunter", # specific fellowship/program in humanities/social sciences
    ],
}

In [263]:
import re

# Prepare expressions for classification
classification_expressions = []
category_columns = list(refined_categories_improved_sorted.keys())

for cat, keywords in refined_categories_improved_sorted.items():
    # Create a regex pattern for all keywords in the category
    # CORRECTED LINE: Use re.escape from the standard 're' module
    pattern = "|".join([f"(?i){re.escape(word)}" for word in keywords]) # (?i) for case-insensitive
    classification_expressions.append(
        # Still use 'node_label' as the target column, as discussed
        pl.when(pl.col('Affiliation_list').str.contains(pattern))
        .then(1)
        .otherwise(0)
        .alias(cat)
    )


# Apply initial classifications to df_exploded
classified_df_exploded = df_exploded.with_columns(
    classification_expressions
)

# Apply interdisciplinary logic in Polars
'''
# Exclude 'Participants_Associations', 'Other', 'Interdisciplinary' from the sum for interdisciplinary check
relevant_category_cols = [col for col in category_columns if col not in ["Participants_Associations", "Other", "Interdisciplinary"]]

classified_df_exploded = classified_df_exploded.with_columns(
    # Set Participants_Associations and Other to 0 if sum of other categories >= 1
    pl.when(pl.sum_horizontal(relevant_category_cols) >= 1)
    .then(0)
    .otherwise(pl.col("Participants_Associations"))
    .alias("Participants_Associations"),

    pl.when(pl.sum_horizontal(relevant_category_cols) >= 1)
    .then(0)
    .otherwise(pl.col("Other"))
    .alias("Other")
)

# Recalculate sums for the final interdisciplinary check
classified_df_exploded = classified_df_exploded.with_columns(
    pl.when(pl.sum_horizontal(relevant_category_cols) > 1)
    .then(1)
    .otherwise(pl.col("Interdisciplinary")) # If it was already 1, keep it 1, otherwise 0
    .alias("Interdisciplinary")
)
'''

print("\nClassified df_exploded (Full Polars):")
#print(classified_df_exploded[['Title', 'Affiliation_list'] + category_columns])


Classified df_exploded (Full Polars):


In [264]:
classified_df_exploded.select(pl.sum(category_columns)).transpose(include_header=True)

column,column_0
str,i32
"""Medical_Sciences""",49662
"""Computer_Science""",41971
"""Engineering""",54441
"""Private_Entity""",42226
"""Interdisciplinary""",78764
"""Participants_Associations""",30373
"""Other""",70565


In [265]:
unique_affiliations_df = df_exploded.select(pl.col("Affiliation_list")).unique()
unique_affiliations_df = unique_affiliations_df.with_columns(
    pl.col("Affiliation_list").str.slice(0, 50).alias("node_label_truncated") # Take first 50 chars
)

classified_unique_affiliations_df = unique_affiliations_df.group_by("node_label_truncated").agg(
    pl.col("Affiliation_list").first().alias("original_node_label") # Get one original for context
).select(
    pl.col("node_label_truncated"),
    pl.col("original_node_label")
)


classified_unique_affiliations_df = unique_affiliations_df.with_columns(
    classification_expressions
)
classified_unique_affiliations_df = classified_unique_affiliations_df.with_columns(
    pl.when(
        pl.sum_horizontal(category_columns) == 0 # Sum across all category flags (0 or 1)
    )
    .then(1) # If sum is 0, it's undetected
    .otherwise(0) # Otherwise, it's detected by at least one category
    .alias("Undetected") # New column name
)
'''
classified_unique_affiliations_df = classified_unique_affiliations_df.with_columns(
    # Set Participants_Associations and Other to 0 if sum of other categories >= 1
    pl.when(pl.sum_horizontal(relevant_category_cols) >= 1)
    .then(0)
    .otherwise(pl.col("Participants_Associations"))
    .alias("Participants_Associations"),

    pl.when(pl.sum_horizontal(relevant_category_cols) >= 1)
    .then(0)
    .otherwise(pl.col("Other"))
    .alias("Other")
)
'''
classified_unique_affiliations_df.select(pl.sum(category_columns)).transpose(include_header=True)

column,column_0
str,i32
"""Medical_Sciences""",41995
"""Computer_Science""",35146
"""Engineering""",44776
"""Private_Entity""",36026
"""Interdisciplinary""",65777
"""Participants_Associations""",25742
"""Other""",58864


In [266]:
result=pl.concat([classified_df_exploded.select(pl.sum(category_columns)),classified_unique_affiliations_df.select(pl.sum(category_columns))])
result=result.transpose(include_header=True)
result = result.rename({
    "column": "Category",
    "column_0": "Count", # Or whatever this column represents
    "column_1": "Unique Affiliations" # Renaming column_1 as requested
})
result = result.with_columns(
    ((pl.col("Count") - pl.col("Unique Affiliations")) / pl.col("Count")*100).round_sig_figs(4).alias("Dif. Percent")
)
# Convert Polars DataFrame to Pandas DataFrame
df_pandas = result.to_pandas()

# Create a Styler object
styler = df_pandas.style

# Apply styling: no index and add caption
styler = styler.hide(axis="index") # Hide the DataFrame index
styler = styler.set_caption("Comparison of Paper Counts and Unique Affiliations by Category")

# Further improvements for LaTeX output (formatting numbers, better rules)
styler = styler.format({
    "Paper Count": "{:,.0f}",          # Format numbers with commas, 0 decimal places
    "Unique Affiliations": "{:,.0f}",   # Format numbers with commas, 0 decimal places
    "Dif. Percent": "{:,.2f}\%"   # Format numbers with commas, 0 decimal places
    
})

# Generate LaTeX with desired options
latex_output = styler.to_latex(
    convert_css=True,
    clines="all;data",       # Add horizontal lines under headers and between data rows
    hrules=True,             # Add \toprule, \midrule, \bottomrule for professional look
    multicol_align="c",      # Alignment for multicolumns (if any)
    environment="longtable" if len(df_pandas) > 10 else "table", # Use longtable for large tables
    position_float="centering", # Center the table on the page
    column_format="l r r r",   # 'l' for Category (left), 'r' for counts (right)
)

print(latex_output)



\begin{table}
\centering
\caption{Comparison of Paper Counts and Unique Affiliations by Category}
\begin{tabular}{l r r r}
\toprule
Category & Count & Unique Affiliations & Dif. Percent \\
\midrule
Medical_Sciences & 49662 & 41,995 & 15.44\% \\
Computer_Science & 41971 & 35,146 & 16.26\% \\
Engineering & 54441 & 44,776 & 17.75\% \\
Private_Entity & 42226 & 36,026 & 14.68\% \\
Interdisciplinary & 78764 & 65,777 & 16.49\% \\
Participants_Associations & 30373 & 25,742 & 15.25\% \\
Other & 70565 & 58,864 & 16.58\% \\
\bottomrule
\end{tabular}
\end{table}



In [267]:
print(classified_df_exploded.head(1))
print(classified_df_exploded[category_columns].head(1))
classified_df_exploded=classified_df_exploded.with_columns(
    pl.concat_str(category_columns).alias("binary_string")
)

shape: (1, 17)
┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────┐
│ Title      ┆ Abstract   ┆ Authors    ┆ Affiliati ┆ … ┆ Private_E ┆ Interdisc ┆ Participa ┆ Other │
│ ---        ┆ ---        ┆ ---        ┆ ons       ┆   ┆ ntity     ┆ iplinary  ┆ nts_Assoc ┆ ---   │
│ str        ┆ str        ┆ str        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ iations   ┆ i32   │
│            ┆            ┆            ┆ str       ┆   ┆ i32       ┆ i32       ┆ ---       ┆       │
│            ┆            ┆            ┆           ┆   ┆           ┆           ┆ i32       ┆       │
╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════╡
│ Deep       ┆ Intracrani ┆ Wu,        ┆ Departmen ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 1     │
│ manifold   ┆ al neural  ┆ Lingyun    ┆ t of Neur ┆   ┆           ┆           ┆           ┆       │
│ learning   ┆ activity   ┆ (572267663 ┆ ology,    ┆   ┆           ┆        

In [274]:
print(classified_df_exploded["binary_string"].value_counts(sort=True).head(10).to_pandas().to_latex(index=False))

\begin{tabular}{lr}
\toprule
binary_string & count \\
\midrule
0111101 & 6839 \\
0000000 & 5757 \\
1111101 & 4344 \\
1000101 & 4201 \\
0110101 & 3914 \\
1010101 & 3778 \\
1011101 & 3107 \\
1111111 & 2646 \\
1010111 & 2460 \\
0010101 & 2449 \\
\bottomrule
\end{tabular}



In [260]:
from collections import Counter
spacy.prefer_gpu()
def analyze_abstract_spacy(abstract_text, num_common_words=10, num_common_phrases=5):
    """
    Analyzes an abstract using spaCy to extract key information.

    Args:
        abstract_text (str): The input abstract string.
        num_common_words (int): Number of most common non-stopword nouns/adjectives/verbs.
        num_common_phrases (int): Number of most common noun chunks.

    Returns:
        dict: A dictionary containing extracted information.
    """
    doc = nlp(abstract_text)

    # --- 1. Get Most Common Words (filtered for key parts of speech and stopwords) ---
    # Filter out stopwords, punctuation, and spaces.
    # Focus on nouns, proper nouns, adjectives, and verbs as they carry more meaning.
    meaningful_tokens = [
        token.lemma_.lower() for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space and
           token.pos_ in ['NOUN', 'PROPN', 'ADJ', 'VERB']
    ]
    most_common_words = Counter(meaningful_tokens).most_common(num_common_words)

    # --- 2. Extract Named Entities (NER) ---
    # Named entities are real-world objects like persons, organizations, locations, etc.
    # These can highlight key actors, technologies, or places mentioned.
    named_entities = []
    for ent in doc.ents:
        if ent.label_ == "ORG": # Add this condition
            named_entities.append(ent.text)
    
    # Optional: Group entities by label for better readability
    entities_by_label = Counter([ent.label_ for ent in doc.ents])
    
    # --- 3. Extract Noun Chunks (Candidate Keyphrases) ---
    # Noun chunks are phrases that have a noun as their head. They often represent key concepts.
    noun_chunks = [chunk.text.lower() for chunk in doc.noun_chunks]
    most_common_noun_chunks = Counter(noun_chunks).most_common(num_common_phrases)

    # --- 4. Identify Potential Subjects/Objects (Dependency Parsing - more advanced) ---
    # This can help infer relationships between concepts.
    # For a simple abstract, we might just look for main subjects.
    main_subjects = []
    for token in doc:
        # A simple heuristic: find nouns that are subjects of verbs
        if "subj" in token.dep_ and token.pos_ in ['NOUN', 'PROPN']:
            main_subjects.append(token.text.lower())
    main_subjects_unique = list(set(main_subjects)) # Unique subjects

    return meaningful_tokens,named_entities,noun_chunks,main_subjects_unique
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm
counter=0
monsters=classified_df_exploded.filter(pl.col("binary_string") == '0000000')

total_abstracts=len(monsters["Affiliation_list"])
batch = []
schema={"meaningful_tokens":pl.List(pl.String), "entities":pl.List(pl.String), "noun_chunks":pl.List(pl.String), "main_subjects_unique":pl.List(pl.String)}
results_lm=pl.DataFrame(schema=schema)
with tqdm(total=total_abstracts) as progress_bar:
    for i in monsters["Affiliation_list"]:
        meaningful_tokens, entities,noun_chunks, main_subjects_unique= analyze_abstract_spacy(str(i)) 
        new_row={"meaningful_tokens": meaningful_tokens, "entities":entities, "noun_chunks":noun_chunks, "main_subjects_unique":main_subjects_unique}   
        batch.append(new_row)
        if counter%1000==0 or counter==total_abstracts-1:
            df_temp=pl.DataFrame(batch,schema=schema,strict=False)
            results_lm.extend(df_temp)
            batch=[] 
        progress_bar.update(1) # update progress
        counter+=1
        #if counter==33927:
            #break


100%|███████████████████████████████████████████████████████████████████████████████| 5760/5760 [01:14<00:00, 77.15it/s]


In [261]:
results_lm.filter(
    (pl.col("entities") != ["none"]) & (pl.col("entities") != [])
)

meaningful_tokens,entities,noun_chunks,main_subjects_unique
list[str],list[str],list[str],list[str]
"[""decine"", ""paris"", ""france""]","[""decine""]","[""decine"", ""france""]",[]
"[""a*star"", ""singapore""]","[""A*Star""]","[""a*star""]",[]
"[""akvaplan"", ""niva"", … ""norway""]","[""Akvaplan-niva""]","[""akvaplan-niva"", ""bergen office"", … ""norway""]",[]
"[""ifhe"", ""shankarapalli"", … ""india""]","[""IFHE""]","[""ifhe"", ""shankarapalli road"", ""india""]",[]


### BREAK

In [ ]:
# Filter for affiliation nodes
#aff_nodes = nodes_df[nodes_df['node_type'] == 'affiliation'].copy()

# Fill any NA values in 'node_label' with an empty string
aff_nodes['node_label'] = aff_nodes['node_label'].fillna('')

# Define your mapping of category names to keywords (all in lowercase)
categories = {
    "Humanity_Science": ["HEC","Philosophy","social","music","culture","history","art"],
    "Social_Science": ["psycholog","Psicolo","tourism","Liser","Management","IMI"],
    "Biology": ["Ahoura Building","Pharma","Microbiology","Physiology","Cytometry","Kinesiology","child development","Bio","Ageing","Genetic","Guangzhou","Eriksholm","Hefei","oil","BGI","3B","Genomic","fMRI","LTCI"],
    "Computer": ["CAPSIX","Cibernetica","LAVID","Fuzzy Inst","ATR","FAST NUCES","CNRS","computer","intel","wireless","tele","systems","ai","smart","Supercomputing","CSE","LIPADE","Data Analysis","Simulation","tech","processing","signal","I.T","IT","Data","LTSI","Iasi","Sys","internet","online","BCI","CDAC","Robotics","Zhejiang Lab","ECE","ICT","Ji Hua Laboratory"],
    "Engineering": ["AJCE","Russell","RIKEN","Budapesti","CSIRO","TU ","CEA","ISEP","IOSB","Schmalkalden","EIE","I.I.E.S.T","IIEST","INEGI","VTU","SSGMCE","FNSPE","JISCE","PIGCE","UIET","JNTU","Hubei Longzhong","TU Braunschweig","TU-Sofia","JEC","Hahn-Schickard","Ingénieur","Inesc","Changsha","Design","UNSTPB","Tianmushan","Delft","eng", "technol","ronics","imec","ele","Physique","Grenoble","Akademie","Techn","Polit","Automation","Physics","Mechanical","Mat","bioengineeri","cybernetics","Biochemistry","Biosciences","DIBRIS","Teknologi","Machine","circuit","ULFG","GRIET","Inria","Analog","Innovative","Inovasi" ,"HHI","VDF GOI","DIEMS"],  # adjust these as needed
    "Private_Entity":["Sulc","WDH","Gedeon Richter","plc","Fraunhofer","Noordwest","Precisis","Sentia","MindRove","LG","Greentek","CSIRO Manufacturing","Whist Lab","Daewon Precision","KaosKey","Ambu A/S, Baltorpbakken","THALES","A*STAR","Charles River","LLC","Ullo","AMO ","Foxstream","Google","Siemens","Everseen","Tecent","Tencent","MindAffect","Porsche","OptoCeutics","HypnoVR","Toyota","VASCage","Hollysys","Soso","Hearing4all","Fujitsu","Zeto","Amazon","IBM","Zalando","Senzhigaoke","Limited","Co","ltd","clinic","GmbH","CESI LINEACT","Inc","sa","Alibaba","R&D","WS Audiology","Solarwinds","International","WeBank","trianect","MindSpire","Epilog","MINDig","group","Byteflies","Pre Diagnostics","Orange","qEEG-Pro","Csl","NEXARCH","Global","SimiaRoom"],
    "Neurology": ["SynapCell","ICM","UDL3","Synaptic","Sommeil","UPMC","Cerebral","Champalimaud","neur","bsi","brain","cognitive","sleep","speech","vision","alzheimer","autismo","CLLE","MEGIN","Mental","CERMEP","Cerveau","CRANIA"],
    "Medicine":["H. Lundbeck","Hangzhou First People","Pfeiffersche","Cirugía","Médica","Neonatology","UZ Leuven","General Practice","Mmu","Médecine","UKM","UMSC","Pediatrics","UMC","Hubei Shizhen","UPC","CHU","thera","med","Inserm","patho","pato","cancer","oncology","radiology","injury","human","health","psiquiatr","psychiatr","hospital","Hôpital","clinic","care","psiquiatr","Anästhesiologie","Anesthesi","restorative","regen","psychiatr","Otorhinolaryngology","sports","epilep","tremo","surgery","Cognition","Cardiovascular","Dementia","Geriatrics","Diabetes","ExpORL"],
    "Interdisciplinary":["Jamia Hamdard","Jamia Millia Islamia","IFHE","CRC","Korea U","HU Berlin","htw","UNSW","RISE","Zhongguancun","Faculdade","seibersdorf","uni","Interdisciplinary","Binghamton","Innovation","research","sciences","college","univer","Science","Independent","LMU","Industries","Cient","Network","Akademi","Academy","CIFAR","CESI","AAU" ],
    "Particiapants_Associations": ["Fonctionnelles","Centre","CASE Danismanlik","footpower","Kinderziekenhuis","Clinique","Troop","spc","Dyslexia","Association","EuroRehab","Pilipili","ESSIQ","CBR Effata","Gymnasium","Klinik","allmyhomes","Abled","Museum","Expert","Orderin","Tennis","Accenture","Hospices","Tam Finans Ar-Ge Tam Finans","Orthopedic Prosthetics","high","school","Foundation","Administrative","Bureau","FHI 360","Promotion","Transport","ambulan","Education","Denken Solution","Timeflux","Residency","center","Ministry","child","Planning","ROC Kop van Noord-Holland"],
    "other":["Alumni","and","Fellow","Serra Hunter","Thiruvananthapuram","CREST","na","CGM",]
}
refined_categories = {
    "Medical_Sciences": ["neur", "brain", "cognitive", "sleep", "speech", "vision", "alzheimer", "autismo", "CLLE", "MEGIN", "Mental", "CERMEP", "Cerveau", "CRANIA", "H. Lundbeck", "Hangzhou First People", "Pfeiffersche", "Cirugía", "Médica", "Neonatology", "UZ Leuven", "General Practice", "Mmu", "Médecine", "UKM", "UMSC", "Pediatrics", "UMC", "Hubei Shizhen", "UPC", "CHU", "thera", "med", "Inserm", "patho", "pato", "cancer", "oncology", "radiology", "injury", "human", "health", "psiquiatr", "psychiatr", "hospital", "Hôpital", "clinic", "care", "Anästhesiologie", "Anesthesi", "restorative", "regen", "Otorhinolaryngology", "sports", "epilep", "tremo", "surgery", "Cognition", "Cardiovascular", "Dementia", "Geriatrics", "Diabetes", "ExpORL"],
    "Computer_Science": ["inform","CAPSIX","Cibernetica","LAVID","Fuzzy Inst","ATR","FAST NUCES","CNRS","computer","intel","wireless","tele","systems","ai","smart","Supercomputing","CSE","LIPADE","Data Analysis","Simulation","tech","processing","signal","I.T ","IT ","Data","LTSI","Iasi","Sys","internet","online","BCI","CDAC","Robotics","Zhejiang Lab","ECE","ICT","Ji Hua Laboratory"],
    "Engineering": ["AJCE","Russell","RIKEN","Budapesti","CSIRO","TU ","CEA","ISEP","IOSB","Schmalkalden","EIE","I.I.E.S.T","IIEST","INEGI","VTU","SSGMCE","FNSPE","JISCE","PIGCE","UIET","JNTU","Hubei Longzhong","TU Braunschweig","TU-Sofia","JEC","Hahn-Schickard","Ingénieur","Inesc","Changsha","Design","UNSTPB","Tianmushan","Delft","eng", "technol","ronics","imec","ele","Physique","Grenoble","Akademie","Techn","Polit","Automation","Physics","Mechanical","Mat","bioengineeri","cybernetics","Biochemistry","Biosciences","DIBRIS","Teknologi","Machine","circuit","ULFG","GRIET","Inria","Analog","Innovative","Inovasi" ,"HHI","VDF GOI","DIEMS"],
    "Private_Entity":["Sulc","WDH","Gedeon Richter","plc","Fraunhofer","Noordwest","Precisis","Sentia","MindRove","LG","Greentek","CSIRO Manufacturing","Whist Lab","Daewon Precision","KaosKey","Ambu A/S, Baltorpbakken","THALES","A*STAR","Charles River","LLC","Ullo","AMO ","Foxstream","Google","Siemens","Everseen","Tecent","Tencent","MindAffect","Porsche","OptoCeutics","HypnoVR","Toyota","VASCage","Hollysys","Soso","Hearing4all","Fujitsu","Zeto","Amazon","IBM","Zalando","Senzhigaoke","Limited","Co","ltd","clinic","GmbH","CESI LINEACT","Inc","sa","Alibaba","R&D","WS Audiology","Solarwinds","International","WeBank","trianect","MindSpire","Epilog","MINDig","group","Byteflies","Pre Diagnostics","Orange","qEEG-Pro","Csl","NEXARCH","Global","SimiaRoom"],
    "Interdisciplinary": ["Jamia Hamdard","Jamia Millia Islamia","IFHE","CRC","Korea U","HU Berlin","htw","UNSW","RISE","Zhongguancun","Faculdade","seibersdorf","uni","Interdisciplinary","Binghamton","Innovation","research","sciences","college","univer","Science","Independent","LMU","Industries","Cient","Network","Akademi","Academy","CIFAR","CESI","AAU" ],
    "Participants_Associations": ["Fonctionnelles","Centre","CASE Danismanlik","footpower","Kinderziekenhuis","Clinique","Troop","spc","Dyslexia","Association","EuroRehab","Pilipili","ESSIQ","CBR Effata","Gymnasium","Klinik","allmyhomes","Abled","Museum","Expert","Orderin","Tennis","Accenture","Hospices","Tam Finans Ar-Ge Tam Finans","Orthopedic Prosthetics","high","school","Foundation","Administrative","Bureau","FHI 360","Promotion","Transport","ambulan","Education","Denken Solution","Timeflux","Residency","center","Ministry","child","Planning","ROC Kop van Noord-Holland"],
    "Other": ["Alumni","and","Fellow","Serra Hunter","Thiruvananthapuram","CREST","na","CGM", "HEC","Philosophy","social","music","culture","history","art","psycholog","Psicolo","tourism","Liser","Management","IMI","Ahoura Building","Pharma","Microbiology","Physiology","Cytometry","Kinesiology","child development","Bio","Ageing","Genetic","Guangzhou","Eriksholm","Hefei","oil","BGI","3B","Genomic","fMRI","LTCI"],
}
# Create a new column for the classification, defaulting to "Other"
#aff_nodes["department_category"] = "Other"
category_columns = list(refined_categories.keys())
for i in  refined_categories.keys():
    aff_nodes[i] = 0 
    
# Loop over the categories:
for cat, keywords in refined_categories.items():
    # Start with a Boolean mask of False for all rows
    for word in keywords:
       
        mask = aff_nodes["node_label"].str.contains(word, case=False, regex=False)
        mask = mask | aff_nodes["node_label"].str.contains(word, case=False,regex=False)

        aff_nodes.loc[mask, cat] = 1
        
category_sums = aff_nodes[category_columns[:-2]].sum(axis=1)
interdisciplinary_mask = category_sums >= 1
aff_nodes.loc[interdisciplinary_mask, "Participants_Associations"] = 0
aff_nodes.loc[interdisciplinary_mask, "Other"] = 0
category_sums = aff_nodes[category_columns[:-2]].sum(axis=1)
interdisciplinary_mask = category_sums > 1
aff_nodes.loc[interdisciplinary_mask, "Interdisciplinary"] = 1



#print(aff_nodes[['node_label', "department_category"]])
aff_nodes.loc[(aff_nodes.Medical_Sciences |aff_nodes.Computer_Science | aff_nodes.Engineering | aff_nodes.Private_Entity | aff_nodes.Interdisciplinary |aff_nodes.Participants_Associations| aff_nodes.Other) == 0]["node_id"]
authors_with_categories =  aff_nodes.join(nodes_df,lsuffix="x")
authored_by_edges=authored_by_edges.rename(columns={"source":"Title"})

In [12]:
# 1. Merge authored_by_edges, affiliation_edges, and aff_nodes.
paper_author_affiliations = authored_by_edges.merge(
    affiliation_edges, left_on='destination', right_on='author', how='left'
).merge(aff_nodes, left_on='affiliation', right_on = 'node_label', how = 'left')

# 2. Filter out rows with missing affiliation information.
paper_author_affiliations_filtered = paper_author_affiliations.dropna(subset=list(refined_categories.keys()))

# 3. Select the required columns.
selected_columns = ['Title'] + list(refined_categories.keys())
paper_category_analysis = paper_author_affiliations_filtered[selected_columns]

# 4. Group by paper and aggregate categories.
paper_category_analysis_grouped = paper_category_analysis.groupby('Title').agg(
    {cat: 'sum' for cat in refined_categories.keys()}
).reset_index()

# Now paper_category_analysis_grouped contains the aggregated category info for each paper.
paper_category_analysis_grouped.head()

,Title,Medical_Sciences,Computer_Science,Engineering,Private_Entity,Interdisciplinary,Participants_Associations,Other
0,Early processing of emotional faces in a Go/No...,0,0,0,0,0,0,0
1,EEG-Based Drowsiness Detection with Fuzzy Inde...,0,4,3,2,3,0,0
2,A unified analytical framework with multiple f...,0,1,1,0,1,0,0
3,Early gesture development as a predictor of au...,43,57,54,31,88,0,0
4,Raman laser intensity and sample clarification...,0,0,0,0,0,0,0


In [24]:

authored_by_edges.loc[authored_by_edges.Title=="Early gesture development as a predictor of autism spectrum disorder in elevated-likelihood infants of ASD"]

,Title,destination,relationship_type
517,Early gesture development as a predictor of au...,Liu L.,authored_by
517,Early gesture development as a predictor of au...,Ye Q.,authored_by
517,Early gesture development as a predictor of au...,Xing Y.,authored_by
517,Early gesture development as a predictor of au...,Xu Y.,authored_by
517,Early gesture development as a predictor of au...,Zhu H.,authored_by
517,Early gesture development as a predictor of au...,Lv S.,authored_by
517,Early gesture development as a predictor of au...,Zou X.,authored_by
517,Early gesture development as a predictor of au...,Deng H.,authored_by
